SELECT DISTINCT
ot.ticket_id as "Id", ot.number as "Ticket Number",
ote.created as "Last Updated",cdta.subject as "Subject", 
ote.body as "Description",ote.poster as "From",sta.name as "Status" , 
ote.type as "type"
FROM ost_ticket AS ot, 
ost_thread AS th, 
ost_thread_entry AS ote , 
ost_ticket_status AS sta, 
ost_ticket__cdata AS cdta
WHERE ote.thread_id = th.id AND 
th.object_id = ot.ticket_id  AND 
ot.status_id = sta.id AND
ot.ticket_id  = cdta.ticket_id AND
sta.name='Closed'  and ote.poster != 'Agent One' and ote.poster != 'Manager One' 
ORDER BY ote.id desc 

In [ ]:
import pandas as pd
import numpy as np
import re
import sqlite3 as sql
import mysql.connector
import unicodedata
import html 

conn = mysql.connector.connect(
   host="db.example.com",
   user="fake_db_user",
   password="FAKE_DB_PASSWORD",
   database="fake_db_name",
   consume_results=True
 )
print(conn)

In [ ]:
df= pd.read_sql(""" 

SELECT DISTINCT
ot.ticket_id as "Id", ot.number as "Ticket Number",
ote.created as "Last Updated",cdta.subject as "Subject", 
ote.body as "Description",ote.poster as "From",sta.name as "Status" , 
ote.type as "Type"
FROM ost_ticket AS ot, 
ost_thread AS th, 
ost_thread_entry AS ote , 
ost_ticket_status AS sta, 
ost_ticket__cdata AS cdta
WHERE ote.thread_id = th.id AND 
th.object_id = ot.ticket_id  AND 
ot.status_id = sta.id AND
ot.ticket_id  = cdta.ticket_id AND
sta.name='Closed' and ote.type !='N'
ORDER BY ote.id desc 

""",conn)

print(df)

# Sözlük yapısı ile kaç tane etiket sınıfımız olduğunu sorguluyoruz kişileri etiketler altında toplayabilelim
set([df.iloc[i]["From"] for i in range (len(df)) if df.iloc[i]["type"]=="R"  and df.iloc[i]["From"] not in list(dictTag.keys())])

In [ ]:
df.info

In [ ]:
dictTag = {
    
    "Manager One":"Yönetim",
    "Agent One":"",
    "M. Halil Akkaynak":"Yazılım",
    "Murat Kısa":"Yazılım",
    "Furkan Kızıltepe":"Yazılım",
    "Melih Kuru":"Yazılım",
    "Selim Ulucan":"Yazılım",
    "Akif Dora":"Yazılım",
    "Elif Küçükaslan":"Teknik Destek",
    "Agent Two":"Teknik Destek",
    "Agent Three":"Teknik Destek",
    "Agent Four":"Teknik Destek",
    "Yasemin Karadağ":"Teknik Destek",
    "Agent Five":"Teknik Destek",
    "Sena Solmaz":"Tasarım",
    "Tugay Öncü":"Tasarım",
    "Ümit Arpat":"Tasarım",
    "Onur Kaplan":"Tasarım",
    "Rabia Ölker":"Tasarım"
}
set([df.iloc[i]["From"] for i in range (len(df)) if df.iloc[i]["Type"]=="R" and df.iloc[i]["From"] not in list(dictTag.keys())])


In [ ]:
# Fonksiyon sayesinden sözlük yapısına ait kişilerin ilgilendiği ticket değerleri görüntülemiş oluyoruz 
dictTicketMessage = {}
for i in range (len(df)):
    ticketNo = df.iloc[i]["Ticket Number"]
    if not ticketNo in dictTicketMessage: dictTicketMessage[ticketNo]=[]
    dictTicketMessage[ticketNo].append(df.iloc[i])

dictTicketMessage.values() #Ticket karşılık gelen verileri görelim
len(dictTicketMessage) # Veri setindeki uzunluğa bakalım
dictTicketMessage.keys() #18792 kayıtta(M-R) oluşuyor
dictTicketMessage['013451'] # herhangi bir kayıta ait Ticket yapılan işlemleri görüntülüyoruz

In [ ]:
# Burda veri seti içindeki son  kayıtlarda bulduğu ilk M ve R kayıtlı verileri alıyor 
# böylece üst üste gelmiş veya gereksiz sözcük dolu metinleri almamış oluyoruz
# Alınan kayıtları tag etiketi altında type R olan ve M olan son -2 kaydı alarak onlara from (kapatan kişi kim ise) etiketi atıyoruz böylece 
# Gelen soruyu cevplayan kişi kim ise etiket olarak (tag) verilmiş  oluyor.
# M-R göre 18792 data içinde lazım olan verileri alalım
arData = []
for ticketNo in list(dictTicketMessage.keys()):
    arMessage = dictTicketMessage[ticketNo][-3:] # dictTicketMessage 3 dizilik bir eleman yeri ayırıyoruz ve konu-gövde-gönderen olarak metni etiket için hazırlıypruz
    if len(arMessage)!=3: continue
    if not (arMessage[0]["type"]=="R" and arMessage[1]["type"]=="M") : continue # 3  kolon üzerinde işlem yapar ve kayıtları alırken M-R göre alacak
    message = arMessage[2]["Description"] #açıklamasına 
    subject = arMessage[1]["Subject"]
    tag = arMessage[0]["From"] # Etiket olarak atanmış ise
    arData.append({"message":message,"subjcet":subject,"tag":tag}) # başına _label_ etiketi olarak vereceğiz
arData

In [ ]:
len(arData) # M-R göre veri aldığımız 18792 data içinden alttaki çıktıyı elde ederiz
arData[10]
columns = ['message','subjcet','tag']
df = pd.DataFrame(arData,columns=columns)
df.dtypes # Kaç tane kolon değerinden oluştuğuna bakıyoruz
df['subjcet'][0:9]
df.head()
df.info()
df['subjcet'][75]

In [ ]:
df

df.isnull().sum().sum()

df.dropna(inplace=True) 

# Data PreProcessing


df["text_clean_message"] = df["message"].apply(lambda x: x.lower())
df.head()


In [ ]:
df["text_clean_message_description"] = df["Description"].apply(lambda x: x.lower())
df["text_clean_message_subject"] = df["Subject"].apply(lambda x: x.lower())
df.head()

In [ ]:
def remove_URL(Description): # description kısmındaki url temizliyoruz

    return re.sub(r"https?://\S+|www\.\S+", "", Description)
# Urls text den çıkarma
df["text_clean_message_description"] = df["text_clean_message_description"].apply(lambda x: remove_URL(x))
df["text_clean_message_description"][31]

In [ ]:
# Remove HTML tags (HTML etiketlerini kadlırma)
# Description ve Subject kısmı için

from bs4 import BeautifulSoup
import re
# same using beautifulsoup
# casefold = lower işlevi görür ingilizce dışındaki diller iç,n geçerlidir.
# beautfiul soup uygulayıp adım adım yeni bir kolon daha ekleyelim
def remove_html_escape(html):
    soup = BeautifulSoup(str(html).strip().casefold(), "lxml")
    text = soup.get_text(separator= " ") # div veya herhangi bir tag metinden çıkarken kelimlerin bitişik olmasını engellemek için " "  koyduk
    text = re.sub(r'[\s\r\n]+',' ',text)
    return text.strip()

a = pd.DataFrame(df)
a['text_clean_message_html'] = a['text_clean_message_description'].apply(remove_html_escape) 
a["text_clean_message_html"]

In [ ]:
a

In [ ]:
def remove_greetings_from_text(text, greetings):
    text_lower = text.casefold()
    for greeting in greetings:
        text = text.replace(greeting, "")
    return text

greetings = [
    "selamlar","sayın","yetkili","değerli","acente2","merhabalar","merhaba",
    "i̇yi günler","çalışmalar","rica etsem","arkadaşlar","bey" ,"görüşmemize","istinaden",
    "kolay gelsin","sayın yetkili","değerli acente","i̇ş ortağımız","ailesi","+ destek@",
    "+ destek@", "sn yetkili","dün telefonla görüşmemize istinaden","tekrardan","tekrar sonsuz teşekkürler",
    "agent1","agent2","agent3","agent4","agent5","agent6","agent7","agent8","agent9",
    "akif","yasin","tugay","murat","furkan","melih","selim","yasemin","sena","hanım","hanim","bey",
    "bu","veya","yada","ya da","ve","class=""msonormal"">"
   ]


# Her bir veri örneği için greetings kelimelerini temizleme işlemi
cleaned_data_set = []
for text in a["text_clean_message_html"]:
    cleaned_text = remove_greetings_from_text(text, greetings)
    cleaned_data_set.append(cleaned_text)

    
# Temizlediğimiz datayı DataFrame olarak bir sutün daha ekleyip yazıralım onun üzerinden işlemlerimize devam edelim
a['text_clean_greeting'] = cleaned_data_set

In [ ]:
a

In [ ]:
def preprocess(text):
    text = re.sub(r'[^\w\s\']',' ', text) # noktalama işaretlerini kaldırmak için
    text = re.sub(r'\s best regards.*', ' ', text) # devamındaki kısımları silmek istersek
    text = re.sub(r'\s saygılarımla.*', ' ', text) # devamındaki kısımları silmek istersek
    text = re.sub(r'\s kimden:.*','',text)
    text = re.sub('\r\n', ' ', text) # \xa0 silmek istersek
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',' ',text) # eposta adresi
  #  text = re.sub(r'\skişisel verilerinizin.*',' ',text)
  #  text = re.sub(r'\sbu mesaj ve ekleri.*',' ',text)
  #  text = re.sub(r'gizlilik notu','',text)
    text = re.sub(r'p\b',' ',text) # metinde kalmış ise p tag alır
    text = re.sub(r'<br\s?\/?>',' ',text) # br tag kaldırma için
    text = re.sub(r'\S+@\S+','', text) # elif@gmail.com vb. ifadeleri kaldırır.
    
 # Telefon numaralarını kaldır # metin içindeki türk telefon numrası formatına ait sayıları siler
 # Bir sürü tel yazma formatı odluğu için bir kaç tane ekliyoruz
    text = re.sub(r'\b\d{10}\b|\b\d{7,12}\b', '', text) 
    text = re.sub(r'\b\d{4}\s\d{3}\s\d{4}\b','',text)
    text = re.sub(r'\b\d{4}\s\d{3}\s\d{2}\s\d{2}\b','',text)
    text = re.sub(r'\b\d{4}\s\d{3}\s\d{1}\s\d{3}\b','',text)
    text = re.sub(r'\b\d{2}\s\d{3}\s\d{3}\s\d{4}\b','',text)



    return text.strip().lower() 

a['text_clean_preprocess'] = a['text_clean_greeting'].map(preprocess)

In [ ]:
a

In [ ]:
a.info

In [ ]:
a.head()

In [ ]:
a['text_clean_preprocess'][48]

In [ ]:


# Sadece belirli kolonların alt kümesini göre veriyi kaydededelim etiketsiz olarak
secilmis_kolonlar = a[['Id', 'Ticket Number','Subject','From','Status','type','text_clean_preprocess']]  # Sadece 'Kolon1' ve 'Kolon3'

secilmis_kolonlar.to_csv('secilmis_kolonlar.csv', index=False)


In [ ]:

#a['text_clean_preprocess'].to_csv("tum.txt")

In [ ]:
a

In [ ]:
a['text_clean_preprocess'].to_csv("text_clean_preprocess.txt")

  text = re.sub(r'\b\w+\s*(?:mah|mahallesi)\.?\s+\w+\s*(?:cad|cadde)\.?\s*[\w\s]+?(\b|\d)\s*(?:mahallesi|cadde)\b','',text)
    text = re.sub(r'\|','',text)
    text = re.sub(r'\b\d{1,3}\s\d{3}\s\d{2}\s\d{2}\b','',text)
    text = re.sub(r'\b\d{4}\s\d{3}\s\d{2}\s\d{2}\b','',text)
    text = re.sub(r'\b\w\b','',text)
    text = re.sub(r'[^\w\s]','',text)
    text = re.sub(r'\s\{\$\S*', '',text) # Süslü parantez içindeki ifadeleri kaldırır
    text = re.sub(r'\n', '', text) # Satır sonu işaretlerini kaldırır
    text = re.sub(r'\(\w*\)', '', text) # Normal parantez içindeki ifadeler silir
    text = re.sub(r'(\W\s)|(\W$)|(\W\d*)', ' ',text) # noktalama işaretlerini kaldırır
    text = re.sub(r'x+((/xx)*/\d*\s*)|x*', '',text)
    text = re.sub(r' +', ' ',text) # Ardışık birden fazla olan boşluğu teke düşürür
    text = re.sub(r'-', '',text) # Gereksiz beyaz boşlukları kaldırın

In [ ]:
a['text_clean_message_html'][491]

In [ ]:
a.dropna(inplace=True) #Eğerki değer içermeyen varsa silmek için
a.shape

 Burda veri seti içindeki son  kayıtlarda bulduğu ilk M ve R kayıtlı verileri alıyor 
 böylece üst üste gelmiş veya gereksiz sözcük dolu metinleri almamış oluyoruz

arData = []
for ticketNo in list(dictTicketMessage.keys()):
    arMessage = dictTicketMessage[ticketNo][-2:] #son iki kayıt yani (M-R) başlangıçta eğitime dahil etmek için alacağız
    if len(arMessage)!=2: continue
    if not (arMessage[0]["type"]=="R" and arMessage[1]["type"]=="M"): continue
    message = arMessage[1]["Description"] #açıklamasına 
    tag = arMessage[0]["From"] # Etiket olarak atanmış ise
    arData.append({"message":message,"tag":tag}) # başına _label_ etiketi olarak vereceğiz
arData

# Test Verilerinde Nan Verileri Silmek 

In [ ]:
a.text_clean_message_html.value_counts()
a['text_clean_message_html'][162]
a.to_csv("ac.csv")

# Eğitim ve Test Veri Etiket Atama 

In [ ]:
# Burda da R tipine göre aldığımız kısımları etiket listesinde karşılık geldiği departman ne ise o etiket altında toplamış olacağız
# Böylece eğitim label olarak sözlük içindeki etiket sayısı azalmış ve onlara göre eğitim yapmış olacağız.
# iloc ile dögüdeki her satır için şart ne ise onu kontrol eder
# iloc[type=R] karşılık gelen değer ile iloc[From] kısmındaki karşılık gelen kısımları eşleştiriyoruz ve dicTag olarak 
# oluşturmuş olduğumuz sözlük içinden karşılığı ne ise eşleştirerek değer ne ise onu getiriyoruz.
# Sözlük yapısı kullanmmaızın nedeni tekrarlı değerlerden kurtulmamız aynı satırda tekrar eden bir değer varsa 1 tanesini alacaktır.

set([(a.iloc[i]["tag"],dictTag[a.iloc[i]["tag"]]) for i in range (len(df)) if a.iloc[i]["tag"] in list(dictTag.keys())])

 # Etiket Tek Bir hanede toplamak için kullanmamız yeterlidir
 set([(df.iloc[i]["tag"],dictTag[df.iloc[i]["tag"]]) for i in range (len(df)) if df.iloc[i]["type"]=="R" and df.iloc[i]["tag"] in list(dictTag.keys())]) 

In [ ]:
#DataFrame'deki her öğenin değerler içinde içerilip içermediği "isin" kontrol ediyoruz loc sadece şarta ait ne varsa onun üzerinden işlem yapıyor.
a['Etiket'] = a['tag'].replace(dictTag)
a.head()

#Eğer etiketleri değiştirmek istersek
#a.Etiket.replace("Teknik Destek","Destek",inplace=True)
#a.Etiket.unique()
#Eğer etiketleri değiştirmek istersek

a.Etiket.value_counts()
a.Etiket.unique()


Gereksiz gördüğümüz sutun kaldıralım ("text_clean_message" verileri temzilenmesini adım adım görmek için kolon eklememiz clean için önemlidir)
columns_to_drop = ["text_clean_message"]
df= df.drop(columns=columns_to_drop,inplace=True)

df['text_clean_description_html'][254]

# Eğitim ve Test Verisi Ayarlama 

In [ ]:
a.info()
a.shape
a.head()

d = ['text_label']
df = df.drop(columns=d)

In [ ]:
a['text_label'] = "__label__" + a['Etiket'].astype(str) + " " + a['text_clean_message_html'].astype(str) 
a['text_subject_label_'] = "__label__" + a['Etiket'].astype(str) + " " + a['text_clean_message_subject'].astype(str) 
a['text_label'][5]
a['text_subject_label_'][5]

In [ ]:
## Kaç tane group
df_grp = a.groupby(['Etiket']).size().reset_index(name='counts')
df_grp
df_grp.describe()
a['text_label'][70]
# Eğitim ve Test Verisi Ayarlama 

c = ["message","tag","text_clean_message"]
df = df.drop(columns=c)

# Eğitim için Kullanılacak Algoritma fastext import ediyoruz
import fasttext

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

#train,test = train_test_split(df,test_size=0.2)

train,test = train_test_split(a,test_size=0.2,random_state =42)


#dosyaları modele göndermeden önce test ve train csv olarak ayırıyor o şekilde gönderiyoruz
train.to_csv("train.csv",columns=["text_label","text_subject_label_"],index=False,header=None)
test.to_csv("test.csv",columns=["text_label","text_subject_label_"],index=False,header=None)

model= fasttext.train_supervised(input="train.csv")
model.test("test.csv")

model.predict("Hem mastertur.az hem de mastertour.az sayfalarında AZ dilinde uçak sayfasında takvim içinde aylar M01, M02 gibi gözüküyor")


# Matrix çizmek için 

In [ ]:
X=a.iloc[:,0:5] # o aralıktaki satır boyu kadar olacak kısımlara kadar olan bilgiyi görüntüle
y=a["Etiket"]
y
random_state = np.random.RandomState(31) #rastgele verilerin gelmesi için   (0 ile 2**32 - 1 )

In [ ]:
#Train ve Test olmak üzere oluşturduğumuz öznitelik verisini bölüyoruz.
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.30,random_state=random_state)
print("X Train: ",len(X_train),"\n","X Test: ",len(X_test),sep="")

#dosyaları modele göndermeden önce test ve train csv olarak ayırıyor o şekilde gönderiyoruz
train.to_csv("trn.csv",columns=["text_label","text_subject_label_"],index=False,header=None)
test.to_csv("tst.csv",columns=["text_label","text_subject_label_"],index=False,header=None)

In [ ]:
# Eğitim için Kullanılacak Algoritma fastext import ediyoruz
import fasttext
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay



model= fasttext.train_supervised(input="trn.csv", lr=0.74, epoch=31, wordNgrams=3) #parametredeki değereler değiştikçe iyileşme oranıda artıyor
data = [model]

result = model.test("tst.csv")
num_samples = result[0]
accuracy = result[1]
recall = result[2]

# Doğruluk hesaplaması

print("Örnek Sayısı:",num_samples,"Accuracy:", accuracy,"Recall:",recall) # Doğruluk:  (19, 0.6842105263157895, 0.6842105263157895) (19 örnek , Accuracy, Recal)


# lr:(Learning Rate - Öğrenme Oranı) her adımda modelin öğrenme ağrılığını güncelleyen bir hyperparameter dir bu oran yüksek olduğunda model hızlı öğrenir oran arttıkça uydurma olasılığı artabilir
# daha az olunca ise öğrenme hızı yavaş olacaktır genelde 0.1 (0.74 max değeri: 0.73 başarı sonucu vermiştir )'den başlatılır,swonuç duruma göre değerler ile oynayabiliriz.

# epoch: Bu ayara kaç kere o veriyi öğrenme sürecini tekrar edecektir deneme yanılma yöntemi ile belirlenebilir.(32-100 arasında düşmeye başladığı için ideal min = 31 değeridir time: score: )
# 50000 epoch için time :1.19.36 sn iken accury : 73.7 iken ; 31 time: 0.01 sn accury: 73.7 o yüzden ideal = 31 olmalıdır. 
                                                                                                               
# wordNgrams: bir bağlam yani diğer kelimeler ile aralarındaki anlam-bağlam ilişkisini temsil eder öğrenme süresini uzatır. küçük veri setlerinde büyütmemek gerekir modelin öğrenmemesine yol açar
# 1 -3 arasındaki başarı oranı max = 3 de accury : 73.7 oldu üstünde bir parametrik değer verildiğinde düştü bu kadar verin için ideal ölçü bunlar (veri seti  arttıkça bu değerlerde değişecektir.)
